# Pneumonia Detection — CNN Training

Trains a small convolutional neural network to classify chest X-rays as
**Normal** or **Pneumonia**, then saves the weights to `cnn_model.h5`.

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/paultimothymooney/chest-xray-pneumonia).
Download it, then set `DATA_DIR` below (or the `DATA_DIR` environment variable) to the
folder that contains the `train/`, `val/`, and `test/` subdirectories.

> Educational project — not a medical device. See `DISCLAIMER.md`.

In [ ]:
import os

# Path to the extracted Kaggle chest_xray dataset (contains train/ val/ test/).
# Override via environment variable, e.g.  export DATA_DIR=/path/to/chest_xray
DATA_DIR = os.environ.get("DATA_DIR", "chest_xray")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
IMG_SIZE = (75, 75)
BATCH_SIZE = 32

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

In [ ]:
x_train = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    color_mode="grayscale", class_mode="binary",
)
x_test = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    color_mode="grayscale", class_mode="binary",
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten

model = Sequential([
    Conv2D(64, (3, 3), activation="relu", input_shape=(75, 75, 1)),
    MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),
    Flatten(),
    Dense(512, activation="relu"),
    Dense(256, activation="relu"),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# Modern Keras: use model.fit (fit_generator was removed) — it accepts generators.
history = model.fit(
    x_train,
    steps_per_epoch=len(x_train),
    epochs=10,
    validation_data=x_test,
    validation_steps=len(x_test),
)

In [ ]:
# 0 -> NORMAL, 1 -> PNEUMONIA
print(x_train.class_indices)

## Single-image inference

`predict_classes()` was removed in TensorFlow 2.6+. Use `model.predict()` and
threshold the sigmoid output at 0.5 instead.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image


def detect(img_path):
    img = keras_image.load_img(img_path, target_size=IMG_SIZE, color_mode="grayscale")
    x = keras_image.img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    prob = float(model.predict(x, verbose=0)[0][0])
    label = "Pneumonia" if prob >= 0.5 else "Normal"
    print(f"{label}  (p={prob:.3f})")
    return label


# Example (edit the path to a real image in your dataset):
# detect(os.path.join(DATA_DIR, "val", "PNEUMONIA", "person1946_bacteria_4874.jpeg"))

In [ ]:
model.save("cnn_model.h5")
print("Saved cnn_model.h5")